# S03 · Sincronización y reducción en GPU (Colab / Kaggle)

> **Cómputo Paralelo · 6BM1 · ESCOM-IPN · 27/1** · Unidad V · Sesión S03 · jueves 26 de noviembre de 2026 · 10:30–12:00 · Salón 3208

**Antes de correr nada, activa la GPU:**

- **Colab:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)* → Guardar.
- **Kaggle:** panel derecho *Settings → Accelerator → GPU* (T4 x2 o P100).

La primera celda debe mostrar una tarjeta NVIDIA. Si no hay GPU disponible, trabaja la correctitud con `NUMBA_ENABLE_CUDASIM=1` en tu laptop (`make test`) y reintenta más tarde.

Sigue el `README.md` de la práctica guiada: **predice por escrito antes de ejecutar cada celda.**

In [ ]:
!nvidia-smi

In [ ]:
# Arquitectura para nvcc: T4 (Colab) -> sm_75; P100 (Kaggle) -> sm_60. Se detecta; si falla, queda sm_75.
import subprocess
ARCH = "sm_75"
try:
    cap = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                                  text=True).strip().splitlines()[0]
    ARCH = "sm_" + cap.replace(".", "")
except Exception as exc:  # sin GPU o nvidia-smi viejo
    print("No se pudo detectar la GPU:", exc)
print("ARCH =", ARCH)

## 1. Escribir los fuentes en el disco de la sesión

Cada celda `%%writefile` guarda un archivo. Son los mismos archivos de `practica-guiada/`; si los editas aquí (por ejemplo, el `TODO(1)` del histograma), vuelve a ejecutar la celda y recompila.

In [ ]:
%%writefile reduce_sum.cu
/*
 * reduce_sum.cu — Reducción (suma) de n floats en la GPU en tres versiones (Unidad V, S03).
 *
 *   atomic      cada hilo hace atomicAdd(out, x[i]) sobre UN float en memoria global:
 *               n operaciones atómicas que compiten por la misma dirección (contención).
 *   tree_div    reducción en árbol por bloque en memoria compartida, stride CRECIENTE y la
 *               condición  if (tid % (2*stride) == 0): los hilos activos quedan intercalados
 *               dentro de cada warp -> DIVERGENCIA en todos los warps en los primeros niveles.
 *   tree_nodiv  árbol con stride DECRECIENTE y  if (tid < stride): los hilos activos son
 *               consecutivos; warps enteros se apagan y solo diverge el warp 0 cuando stride < 32.
 *
 *   Las versiones de árbol dejan UNA suma parcial por bloque. Segunda etapa:
 *     etapa2=cpu     el anfitrión copia los gridDim parciales y los suma en double (0 atómicas)
 *     etapa2=atomic  el hilo 0 de cada bloque hace atomicAdd(out, parcial)     (gridDim atómicas)
 *
 * Uso:  ./reduce_sum [version] [n] [block] [etapa2] [archivo.csv]
 *   version  atomic | tree_div | tree_nodiv | all        (por defecto all)
 *   n        elementos (por defecto 1 048 576 = 2^20, 4 MB)
 *   block    hilos por bloque, potencia de 2 <= 1024      (por defecto 256)
 *   etapa2   cpu | atomic                                 (por defecto cpu)
 *   csv      si se da, agrega una línea por versión (lo usa el notebook)
 *
 * Datos (iguales en reduce_sum_cpu.c y reduce_sum_numba.py):
 *   x[i] = 1.0f + (i % 10) * 0.1f   y   x[n-1] = 9.5f  (un máximo único al final, para la fase Modificar)
 *   Referencia: suma secuencial en double. Como 0.1 no es exacto en float y el orden de las sumas
 *   cambia entre versiones, la comparación se hace con tolerancia relativa (TOL_REL).
 *
 * Compilar y correr (Colab/Kaggle con GPU):   nvcc -O2 -arch=sm_75 reduce_sum.cu -o reduce_sum && ./reduce_sum
 * Verificar que compila sin GPU:              nvcc -c -arch=sm_75 reduce_sum.cu -o reduce_sum.o
 * Experimento de la fase Investigar (carrera entre niveles del árbol):
 *   nvcc -O2 -arch=sm_75 -DSIN_SYNC reduce_sum.cu -o reduce_sum_sin_sync
 *
 * Medición: protocolo del curso (calentamiento + 5 repeticiones, mediana) con cudaEvent_t.
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

#define REPS 5
#define MAX_BLOCK 1024          /* máximo de hilos por bloque en cualquier GPU actual */
#define TOL_REL 1e-4            /* tolerancia relativa: ver README, sección "Tolerancia" */

#define CUDA_CHECK(call)                                                           \
    do {                                                                           \
        cudaError_t err__ = (call);                                                \
        if (err__ != cudaSuccess) {                                                \
            fprintf(stderr, "Error de CUDA en %s:%d: %s\n", __FILE__, __LINE__,   \
                    cudaGetErrorString(err__));                                    \
            exit(EXIT_FAILURE);                                                    \
        }                                                                          \
    } while (0)

/* ---------- Código de dispositivo ---------- */

/* Versión 1: atómica ingenua. Correcta, pero las n sumas se serializan sobre una dirección. */
__global__ void reduce_atomic(const float *x, float *out, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        atomicAdd(out, x[i]);   /* lectura-suma-escritura indivisible; el orden de llegada es arbitrario */
}

/* Versión 2: árbol CON divergencia. Nivel 1 activa los hilos pares, nivel 2 los múltiplos de 4...
 * En cada warp de 32 hilos quedan activos 16, 8, 4, 2, 1: el warp ejecuta el if con lanes apagados. */
__global__ void reduce_tree_div(const float *x, float *partial, float *out, int n, int atomic_stage2)
{
    __shared__ float s[MAX_BLOCK];          /* memoria compartida: visible solo dentro del bloque */
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;

    s[tid] = (i < n) ? x[i] : 0.0f;         /* carga cooperativa; fuera de rango carga el neutro 0 */
    __syncthreads();                        /* (A) nadie suma hasta que TODO el bloque cargó */

    for (int stride = 1; stride < blockDim.x; stride *= 2) {
        if (tid % (2 * stride) == 0)        /* activos: 0, 2s, 4s, ... intercalados en cada warp */
            s[tid] += s[tid + stride];
#ifndef SIN_SYNC
        __syncthreads();                    /* (B) el nivel completo termina antes del siguiente */
#endif
    }
    if (tid == 0) {                         /* s[0] tiene la suma del bloque */
        if (atomic_stage2)
            atomicAdd(out, s[0]);           /* etapa 2 en GPU: una atómica por bloque */
        else
            partial[blockIdx.x] = s[0];     /* etapa 2 en CPU: el anfitrión suma los parciales */
    }
}

/* Versión 3: árbol SIN divergencia (hasta stride < 32). Nivel 1 activa los hilos 0..127 (warps 0-3
 * completos; warps 4-7 completos pero apagados), nivel 2 los hilos 0..63, etc. */
__global__ void reduce_tree_nodiv(const float *x, float *partial, float *out, int n, int atomic_stage2)
{
    __shared__ float s[MAX_BLOCK];
    int tid = threadIdx.x;
    int i = blockIdx.x * blockDim.x + tid;

    s[tid] = (i < n) ? x[i] : 0.0f;
    __syncthreads();                        /* (A) */

    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)                   /* activos: 0..stride-1, consecutivos */
            s[tid] += s[tid + stride];
#ifndef SIN_SYNC
        __syncthreads();                    /* (B) */
#endif
    }
    if (tid == 0) {
        if (atomic_stage2)
            atomicAdd(out, s[0]);
        else
            partial[blockIdx.x] = s[0];
    }
}

/* ---------- Código de anfitrión ---------- */

static void init_data(float *x, int n)
{
    for (int i = 0; i < n; i++)
        x[i] = 1.0f + (float)(i % 10) * 0.1f;
    x[n - 1] = 9.5f;                        /* máximo único al final: la fase Modificar lo usa */
}

/* Referencia: suma secuencial en double (sin pérdida apreciable para n <= 2^24). */
static double sum_double(const float *x, int n)
{
    double acc = 0.0;
    for (int i = 0; i < n; i++)
        acc += x[i];
    return acc;
}

/* Emulación en CPU, en float, del MISMO orden de sumas que cada kernel en árbol. Si el kernel
 * está bien, su parcial por bloque coincide bit a bit con esta emulación (misma aritmética IEEE,
 * mismo orden). Sirve para ver que la diferencia con la referencia es solo el orden, no un error. */
static void tree_emulation_cpu(const float *x, int n, int block, int divergent, float *partial_out)
{
    static float s[MAX_BLOCK];
    int grid = (n + block - 1) / block;
    for (int b = 0; b < grid; b++) {
        for (int tid = 0; tid < block; tid++) {
            int i = b * block + tid;
            s[tid] = (i < n) ? x[i] : 0.0f;
        }
        if (divergent) {
            for (int stride = 1; stride < block; stride *= 2)
                for (int tid = 0; tid < block; tid++)
                    if (tid % (2 * stride) == 0)
                        s[tid] += s[tid + stride];
        } else {
            for (int stride = block / 2; stride > 0; stride >>= 1)
                for (int tid = 0; tid < stride; tid++)
                    s[tid] += s[tid + stride];
        }
        partial_out[b] = s[0];
    }
}

static int cmp_float(const void *a, const void *b)
{
    float fa = *(const float *)a, fb = *(const float *)b;
    return (fa > fb) - (fa < fb);
}

static float median_ms(float *v, int k)
{
    qsort(v, (size_t)k, sizeof(float), cmp_float);
    return v[k / 2];
}

static double now_ms(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec * 1e3 + (double)ts.tv_nsec / 1e6;
}

static int is_power_of_two(int v)
{
    return v > 0 && (v & (v - 1)) == 0;
}

/* Ejecuta una versión con el protocolo de medición y la verifica. Regresa 1 si pasa. */
static int run_version(const char *version, const float *d_x, float *d_partial, float *d_out,
                       const float *h_x, int n, int block, int grid, int atomic_stage2,
                       double ref, const char *gpu_name, const char *csv_path)
{
    int is_atomic = strcmp(version, "atomic") == 0;
    int is_div = strcmp(version, "tree_div") == 0;
    float *h_partial = (float *)malloc((size_t)grid * sizeof(float));
    float *h_emul = (float *)malloc((size_t)grid * sizeof(float));
    if (!h_partial || !h_emul) {
        fprintf(stderr, "Sin memoria en el anfitrión\n");
        exit(EXIT_FAILURE);
    }

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float t_kernel[REPS];
    double sum = 0.0, t_stage2_ms = 0.0;
    long long n_atomics = is_atomic ? n : (atomic_stage2 ? grid : 0);

    /* Calentamiento + REPS repeticiones. En cada una se reinicia el acumulador. */
    for (int r = -1; r < REPS; r++) {
        CUDA_CHECK(cudaMemset(d_out, 0, sizeof(float)));
        CUDA_CHECK(cudaEventRecord(start));
        if (is_atomic)
            reduce_atomic<<<grid, block>>>(d_x, d_out, n);
        else if (is_div)
            reduce_tree_div<<<grid, block>>>(d_x, d_partial, d_out, n, atomic_stage2);
        else
            reduce_tree_nodiv<<<grid, block>>>(d_x, d_partial, d_out, n, atomic_stage2);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaGetLastError());          /* ¿falló el lanzamiento? */
        CUDA_CHECK(cudaEventSynchronize(stop));  /* espera; aquí aparecen los errores de ejecución */
        if (r >= 0)
            CUDA_CHECK(cudaEventElapsedTime(&t_kernel[r], start, stop));
    }

    /* Resultado final de la última repetición */
    if (is_atomic || atomic_stage2) {
        float h_out = 0.0f;
        CUDA_CHECK(cudaMemcpy(&h_out, d_out, sizeof(float), cudaMemcpyDeviceToHost));
        sum = (double)h_out;
    } else {
        double t0 = now_ms();
        CUDA_CHECK(cudaMemcpy(h_partial, d_partial, (size_t)grid * sizeof(float), cudaMemcpyDeviceToHost));
        for (int b = 0; b < grid; b++)           /* etapa 2 en CPU, en double */
            sum += h_partial[b];
        t_stage2_ms = now_ms() - t0;
    }

    /* Verificación contra la CPU con tolerancia relativa */
    double err_rel = fabs(sum - ref) / fabs(ref);
    int ok = (err_rel <= TOL_REL);

    /* Para los árboles con etapa 2 en CPU: ¿coinciden los parciales bit a bit con la emulación? */
    int bitwise = -1;
    if (!is_atomic && !atomic_stage2) {
        tree_emulation_cpu(h_x, n, block, is_div, h_emul);
        bitwise = (memcmp(h_partial, h_emul, (size_t)grid * sizeof(float)) == 0);
    }

    float k_med = median_ms(t_kernel, REPS), k_min = t_kernel[0], k_max = t_kernel[REPS - 1];
    printf("version=%-10s etapa2=%-6s atomicas_globales=%-8lld kernel_mediana_ms=%.4f (min %.4f, max %.4f)",
           version, is_atomic ? "-" : (atomic_stage2 ? "atomic" : "cpu"), n_atomics, k_med, k_min, k_max);
    if (!is_atomic && !atomic_stage2)
        printf(" etapa2_cpu_ms=%.4f", t_stage2_ms);
    printf("\n   suma=%.6f err_rel=%.2e verificacion=%s", sum, err_rel, ok ? "OK" : "FALLA");
    if (bitwise >= 0)
        printf(" parciales_bit_a_bit_igual_a_emulacion_cpu=%s", bitwise ? "si" : "no");
    printf("\n");

    if (csv_path) {
        FILE *f = fopen(csv_path, "a");
        if (f) {
            fseek(f, 0, SEEK_END);
            if (ftell(f) == 0)
                fprintf(f, "gpu,version,etapa2,n,block,grid,atomicas_globales,kernel_mediana_ms,"
                           "kernel_min_ms,kernel_max_ms,etapa2_cpu_ms,suma,ref_double,err_rel,verificacion\n");
            fprintf(f, "\"%s\",%s,%s,%d,%d,%d,%lld,%.4f,%.4f,%.4f,%.4f,%.6f,%.6f,%.3e,%s\n", gpu_name, version,
                    is_atomic ? "-" : (atomic_stage2 ? "atomic" : "cpu"), n, block, grid, n_atomics, k_med, k_min,
                    k_max, t_stage2_ms, sum, ref, err_rel, ok ? "OK" : "FALLA");
            fclose(f);
        }
    }

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    free(h_partial);
    free(h_emul);
    return ok;
}

int main(int argc, char **argv)
{
    const char *version = (argc > 1) ? argv[1] : "all";
    int n = (argc > 2) ? atoi(argv[2]) : (1 << 20);
    int block = (argc > 3) ? atoi(argv[3]) : 256;
    const char *stage2 = (argc > 4) ? argv[4] : "cpu";
    const char *csv_path = (argc > 5) ? argv[5] : NULL;

    int run_atomic = strcmp(version, "all") == 0 || strcmp(version, "atomic") == 0;
    int run_div = strcmp(version, "all") == 0 || strcmp(version, "tree_div") == 0;
    int run_nodiv = strcmp(version, "all") == 0 || strcmp(version, "tree_nodiv") == 0;
    int atomic_stage2 = strcmp(stage2, "atomic") == 0;

    if ((!run_atomic && !run_div && !run_nodiv) || n <= 0 || block <= 0 || block > MAX_BLOCK ||
        (strcmp(stage2, "cpu") != 0 && !atomic_stage2)) {
        fprintf(stderr, "uso: %s [atomic|tree_div|tree_nodiv|all] [n>0] [block 1..%d] [cpu|atomic] [archivo.csv]\n",
                argv[0], MAX_BLOCK);
        return 1;
    }
    if ((run_div || run_nodiv) && !is_power_of_two(block))
        fprintf(stderr, "aviso: el árbol supone block potencia de 2; con block=%d se pierden sumandos "
                        "(experimento de la fase Investigar)\n", block);

    int grid = (n + block - 1) / block;
    size_t bytes = (size_t)n * sizeof(float);

    float *h_x = (float *)malloc(bytes);
    if (!h_x) {
        fprintf(stderr, "Sin memoria en el anfitrión para n=%d\n", n);
        return 1;
    }
    init_data(h_x, n);
    double ref = sum_double(h_x, n);

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("reduce_sum (CUDA C): n=%d block=%d grid=%d gpu=\"%s\" (SMs=%d, cc %d.%d, warp=%d)\n", n, block, grid,
           prop.name, prop.multiProcessorCount, prop.major, prop.minor, prop.warpSize);
    printf("referencia_cpu_double=%.6f  tolerancia_relativa=%.0e\n", ref, TOL_REL);
#ifdef SIN_SYNC
    printf("AVISO: compilado con -DSIN_SYNC: el árbol NO sincroniza entre niveles (carrera deliberada)\n");
#endif

    float *d_x = NULL, *d_partial = NULL, *d_out = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_x, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_partial, (size_t)grid * sizeof(float)));
    CUDA_CHECK(cudaMalloc((void **)&d_out, sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, h_x, bytes, cudaMemcpyHostToDevice));

    int all_ok = 1;
    if (run_atomic)
        all_ok &= run_version("atomic", d_x, d_partial, d_out, h_x, n, block, grid, 0, ref, prop.name, csv_path);
    if (run_div)
        all_ok &= run_version("tree_div", d_x, d_partial, d_out, h_x, n, block, grid, atomic_stage2, ref,
                              prop.name, csv_path);
    if (run_nodiv)
        all_ok &= run_version("tree_nodiv", d_x, d_partial, d_out, h_x, n, block, grid, atomic_stage2, ref,
                              prop.name, csv_path);

    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_partial));
    CUDA_CHECK(cudaFree(d_out));
    free(h_x);
    return all_ok ? 0 : 2;
}

In [ ]:
%%writefile reduce_sum_cpu.c
/*
 * reduce_sum_cpu.c — Referencia secuencial de reduce_sum.cu (Unidad V, S03).
 *
 * Calcula la misma suma de cuatro maneras para mostrar que, en float, el ORDEN de las sumas
 * cambia el resultado (idea errónea 10), y emula el orden exacto de los kernels en árbol:
 *   ref_double            suma secuencial en double: la referencia contra la que se verifica
 *   float_secuencial      suma secuencial en float (i = 0, 1, 2, ...)
 *   float_orden_permutado suma en float recorriendo los índices en otro orden (emula el orden
 *                         arbitrario en que llegan las atómicas a la GPU)
 *   float_arbol_div       mismo orden que reduce_tree_div (árbol por bloque) + etapa 2 en double
 *   float_arbol_nodiv     mismo orden que reduce_tree_nodiv + etapa 2 en double
 * También imprime el máximo (fase Modificar) y cuántas operaciones atómicas haría cada versión
 * de GPU con ese n y ese block (fase Predecir).
 *
 * Uso:   ./reduce_sum_cpu [n] [block]        (por defecto 1048576 256)
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra reduce_sum_cpu.c -o reduce_sum_cpu -lm
 * Datos: x[i] = 1.0f + (i % 10) * 0.1f, x[n-1] = 9.5f  (iguales a reduce_sum.cu y reduce_sum_numba.py)
 */
#define _POSIX_C_SOURCE 199309L   /* clock_gettime con -std=c11 */
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define REPS 5
#define MAX_BLOCK 1024
#define TOL_REL 1e-4

static void init_data(float *x, int n)
{
    for (int i = 0; i < n; i++)
        x[i] = 1.0f + (float)(i % 10) * 0.1f;
    x[n - 1] = 9.5f;
}

static double sum_double(const float *x, int n)
{
    double acc = 0.0;
    for (int i = 0; i < n; i++)
        acc += x[i];
    return acc;
}

static float sum_float_sequential(const float *x, int n)
{
    float acc = 0.0f;
    for (int i = 0; i < n; i++)
        acc += x[i];
    return acc;
}

static long gcd(long a, long b)
{
    while (b != 0) {
        long t = a % b;
        a = b;
        b = t;
    }
    return a;
}

/* Recorre los n índices en el orden j, 2j, 3j, ... (mod n) con j coprimo con n: una permutación
 * determinista que imita el orden arbitrario de llegada de las atómicas. */
static float sum_float_permuted(const float *x, int n)
{
    long step = 1000003;
    while (gcd(step, n) != 1)
        step += 2;
    float acc = 0.0f;
    long idx = 0;
    for (int k = 0; k < n; k++) {
        acc += x[idx];
        idx = (idx + step) % n;
    }
    return acc;
}

/* Emula, en float y en el mismo orden, la reducción en árbol por bloque de los kernels. */
static double sum_tree_emulation(const float *x, int n, int block, int divergent)
{
    static float s[MAX_BLOCK];
    int grid = (n + block - 1) / block;
    double total = 0.0;                 /* etapa 2 en double, como el anfitrión de reduce_sum.cu */
    for (int b = 0; b < grid; b++) {
        for (int tid = 0; tid < block; tid++) {
            int i = b * block + tid;
            s[tid] = (i < n) ? x[i] : 0.0f;
        }
        if (divergent) {
            for (int stride = 1; stride < block; stride *= 2)
                for (int tid = 0; tid < block; tid++)
                    if (tid % (2 * stride) == 0)
                        s[tid] += s[tid + stride];
        } else {
            for (int stride = block / 2; stride > 0; stride >>= 1)
                for (int tid = 0; tid < stride; tid++)
                    s[tid] += s[tid + stride];
        }
        total += s[0];
    }
    return total;
}

static double now_ms(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec * 1e3 + (double)ts.tv_nsec / 1e6;
}

static int cmp_double(const void *a, const void *b)
{
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

static int report(const char *name, double value, double ref, const char *note)
{
    double err_rel = fabs(value - ref) / fabs(ref);
    int ok = err_rel <= TOL_REL;
    printf("%-22s=%.6f  err_rel=%.2e  %s%s\n", name, value, err_rel, ok ? "dentro_de_tolerancia" : "FUERA", note);
    return ok;
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 20);
    int block = (argc > 2) ? atoi(argv[2]) : 256;
    if (n <= 0 || block <= 0 || block > MAX_BLOCK) {
        fprintf(stderr, "uso: %s [n>0] [block 1..%d]\n", argv[0], MAX_BLOCK);
        return 1;
    }
    int grid = (n + block - 1) / block;
    int pow2 = (block & (block - 1)) == 0;

    float *x = (float *)malloc((size_t)n * sizeof(float));
    if (!x) {
        fprintf(stderr, "Sin memoria para n=%d\n", n);
        return 1;
    }
    init_data(x, n);

    /* Tiempo de la referencia (protocolo: calentamiento + REPS, mediana) */
    double t[REPS];
    volatile double sink = sum_double(x, n);    /* calentamiento */
    for (int r = 0; r < REPS; r++) {
        double t0 = now_ms();
        sink = sum_double(x, n);
        t[r] = now_ms() - t0;
    }
    qsort(t, REPS, sizeof(double), cmp_double);
    double ref = sink;

    printf("reduce_sum_cpu: n=%d block=%d grid=%d\n", n, block, grid);
    printf("ref_double            =%.6f  (suma secuencial en double; mediana_ms=%.3f de %d, min %.3f, max %.3f)\n",
           ref, t[REPS / 2], REPS, t[0], t[REPS - 1]);

    /* Las dos sumas en float con un solo acumulador son informativas: muestran el efecto del orden y, con
     * n grande (2^24), la saturación del acumulador (sumar 1.0 a 16 777 216 ya no cambia el float).
     * La verificación del programa cubre las emulaciones del árbol (son lo que hace la GPU) y el máximo. */
    int ok = 1;
    report("float_secuencial", (double)sum_float_sequential(x, n), ref, "  (informativo: un acumulador)");
    report("float_orden_permutado", (double)sum_float_permuted(x, n), ref,
           "  (informativo: emula el orden arbitrario de las atómicas)");
    if (pow2) {
        ok &= report("float_arbol_div", sum_tree_emulation(x, n, block, 1), ref, "  (mismo orden que tree_div)");
        ok &= report("float_arbol_nodiv", sum_tree_emulation(x, n, block, 0), ref, "  (mismo orden que tree_nodiv)");
    } else {
        printf("float_arbol_*         : no se emulan; block=%d no es potencia de 2 (el árbol perdería sumandos)\n",
               block);
    }

    /* Máximo y su posición (fase Modificar: reducción max) */
    float mx = x[0];
    int imx = 0;
    for (int i = 1; i < n; i++)
        if (x[i] > mx) {
            mx = x[i];
            imx = i;
        }
    printf("max=%.6f en i=%d\n", mx, imx);
    ok &= (mx == 9.5f && imx == n - 1);

    /* Operaciones atómicas en memoria global que haría cada versión de GPU (fase Predecir) */
    printf("atomicas_globales: atomic=%d  tree+etapa2=cpu: 0  tree+etapa2=atomic: %d\n", n, grid);
    printf("verificacion=%s  (emulaciones del árbol dentro de tolerancia y máximo correcto)\n", ok ? "OK" : "FALLA");

    free(x);
    return ok ? 0 : 2;
}

In [ ]:
%%writefile reduce_sum_numba.py
#!/usr/bin/env python3
"""reduce_sum_numba.py — Espejo en Numba CUDA de los tres kernels de reduce_sum.cu.

Uso:
    python3 reduce_sum_numba.py [version] [n] [block] [--etapa2 cpu|atomic] [--sin-sync]
      version: atomic | tree_div | tree_nodiv | all   (por defecto all)
Sin GPU (simulador en CPU, solo correctitud; n pequeño):
    NUMBA_ENABLE_CUDASIM=1 python3 reduce_sum_numba.py all 1000 256

Mismos datos, mismo orden de sumas y misma verificación que la versión en C:
    x[i] = 1.0 + (i % 10) * 0.1 (float32), x[n-1] = 9.5; referencia en double; tolerancia relativa 1e-4.
--sin-sync quita la cuda.syncthreads() de dentro del ciclo del árbol (carrera deliberada, fase Investigar).
"""
import argparse
import os
import sys
import time

import numpy as np
from numba import cuda, float32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5
TOL_REL = 1e-4


@cuda.jit
def reduce_atomic(x, out, n):
    # Versión 1: cada hilo suma su elemento sobre UNA posición de memoria global.
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        cuda.atomic.add(out, 0, x[i])


def make_tree_kernels(block, sync_levels=True):
    """Los arreglos en memoria compartida necesitan tamaño constante: se captura `block` en la clausura."""

    @cuda.jit
    def reduce_tree_div(x, partial, out, n, atomic_stage2):
        # Versión 2: árbol CON divergencia (stride creciente, hilos activos intercalados).
        s = cuda.shared.array(block, float32)
        tid = cuda.threadIdx.x
        i = cuda.blockIdx.x * cuda.blockDim.x + tid
        s[tid] = x[i] if i < n else float32(0.0)
        cuda.syncthreads()                      # (A) todo el bloque cargó
        stride = 1
        while stride < cuda.blockDim.x:
            if tid % (2 * stride) == 0:
                s[tid] += s[tid + stride]
            if sync_levels:
                cuda.syncthreads()              # (B) nivel completo antes del siguiente
            stride *= 2
        if tid == 0:
            if atomic_stage2:
                cuda.atomic.add(out, 0, s[0])
            else:
                partial[cuda.blockIdx.x] = s[0]

    @cuda.jit
    def reduce_tree_nodiv(x, partial, out, n, atomic_stage2):
        # Versión 3: árbol SIN divergencia (stride decreciente, hilos activos consecutivos).
        s = cuda.shared.array(block, float32)
        tid = cuda.threadIdx.x
        i = cuda.blockIdx.x * cuda.blockDim.x + tid
        s[tid] = x[i] if i < n else float32(0.0)
        cuda.syncthreads()                      # (A)
        stride = cuda.blockDim.x // 2
        while stride > 0:
            if tid < stride:
                s[tid] += s[tid + stride]
            if sync_levels:
                cuda.syncthreads()              # (B)
            stride //= 2
        if tid == 0:
            if atomic_stage2:
                cuda.atomic.add(out, 0, s[0])
            else:
                partial[cuda.blockIdx.x] = s[0]

    return reduce_tree_div, reduce_tree_nodiv


def init_data(n):
    # Misma aritmética en float32, paso a paso, que `1.0f + (float)(i % 10) * 0.1f` en C: así los datos son
    # idénticos bit a bit en los tres programas (calcular en double y redondear una vez daría otros valores).
    k = (np.arange(n) % 10).astype(np.float32)
    x = np.float32(1.0) + k * np.float32(0.1)
    x[n - 1] = np.float32(9.5)
    return x


def tree_emulation_cpu(x, n, block, divergent):
    """Mismo orden de sumas que el kernel, en float32, vectorizado por bloque (NumPy)."""
    grid = (n + block - 1) // block
    s = np.zeros((grid, block), dtype=np.float32)
    s.flat[:n] = x
    if divergent:
        stride = 1
        while stride < block:
            idx = np.arange(0, block, 2 * stride)
            s[:, idx] += s[:, idx + stride]
            stride *= 2
    else:
        stride = block // 2
        while stride > 0:
            s[:, :stride] += s[:, stride:2 * stride]
            stride //= 2
    return s[:, 0].copy()


def median(values):
    return sorted(values)[len(values) // 2]


def run_version(name, kernel, d_x, d_partial, d_out, x, n, block, grid, atomic_stage2, ref):
    is_atomic = name == "atomic"
    times_ms = []
    # GPU: calentamiento (incluye el JIT) + REPS repeticiones. CUDASIM: un solo lanzamiento; el tiempo del
    # simulador no es una medición y cada lanzamiento de un árbol tarda segundos.
    reps = range(0, 1) if CUDASIM else range(-1, REPS)
    for r in reps:
        d_out[0] = 0.0
        t0 = time.perf_counter()
        if is_atomic:
            kernel[grid, block](d_x, d_out, n)
        else:
            kernel[grid, block](d_x, d_partial, d_out, n, atomic_stage2)
        cuda.synchronize()
        if r >= 0:
            times_ms.append((time.perf_counter() - t0) * 1e3)

    if is_atomic or atomic_stage2:
        total = float(d_out.copy_to_host()[0])
        n_atomics = n if is_atomic else grid
        stage2 = "-" if is_atomic else "atomic"
    else:
        partial = d_partial.copy_to_host()
        total = float(np.sum(partial, dtype=np.float64))   # etapa 2 en CPU, en double
        n_atomics = 0
        stage2 = "cpu"

    err_rel = abs(total - ref) / abs(ref)
    ok = err_rel <= TOL_REL
    if CUDASIM:
        timing = f"tiempo_simulador_ms={times_ms[0]:.1f} (NO es una medición)"
    else:
        timing = f"kernel_mediana_ms={median(times_ms):.4f} (min {min(times_ms):.4f}, max {max(times_ms):.4f})"
    print(f"version={name:<10} etapa2={stage2:<6} atomicas_globales={n_atomics:<8} {timing}")
    line = f"   suma={total:.6f} err_rel={err_rel:.2e} verificacion={'OK' if ok else 'FALLA'}"
    if not is_atomic and not atomic_stage2:
        emul = tree_emulation_cpu(x, n, block, name == "tree_div")
        same = np.array_equal(partial, emul)
        line += f" parciales_bit_a_bit_igual_a_emulacion_cpu={'si' if same else 'no'}"
    print(line)
    return ok


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("version", nargs="?", default="all", choices=["atomic", "tree_div", "tree_nodiv", "all"])
    parser.add_argument("n", nargs="?", type=int, default=(1000 if CUDASIM else 1 << 20))
    parser.add_argument("block", nargs="?", type=int, default=256)
    parser.add_argument("--etapa2", choices=["cpu", "atomic"], default="cpu")
    parser.add_argument("--sin-sync", action="store_true", help="quita la syncthreads entre niveles del árbol")
    args = parser.parse_args()
    n, block = args.n, args.block
    if n <= 0 or block <= 0 or block > 1024:
        sys.exit("uso: reduce_sum_numba.py [version] [n>0] [block 1..1024] [--etapa2 cpu|atomic] [--sin-sync]")
    if CUDASIM and n > 20000:
        print("aviso: CUDASIM con n > 20000 puede tardar varios minutos", file=sys.stderr)
    if args.version != "atomic" and (block & (block - 1)) != 0:
        print(f"aviso: el árbol supone block potencia de 2; con block={block} se pierden sumandos "
              "(experimento de la fase Investigar)", file=sys.stderr)

    grid = (n + block - 1) // block
    atomic_stage2 = args.etapa2 == "atomic"
    x = init_data(n)
    ref = float(np.sum(x, dtype=np.float64))

    d_x = cuda.to_device(x)
    d_partial = cuda.to_device(np.zeros(grid, dtype=np.float32))
    d_out = cuda.to_device(np.zeros(1, dtype=np.float32))

    print(f"reduce_sum_numba: n={n} block={block} grid={grid} modo={'CUDASIM' if CUDASIM else 'GPU'}")
    print(f"referencia_cpu_double={ref:.6f}  tolerancia_relativa={TOL_REL:.0e}")
    if args.sin_sync:
        print("AVISO: --sin-sync: el árbol NO sincroniza entre niveles (carrera deliberada)")

    tree_div, tree_nodiv = make_tree_kernels(block, sync_levels=not args.sin_sync)
    todo = []
    if args.version in ("all", "atomic"):
        todo.append(("atomic", reduce_atomic))
    if args.version in ("all", "tree_div"):
        todo.append(("tree_div", tree_div))
    if args.version in ("all", "tree_nodiv"):
        todo.append(("tree_nodiv", tree_nodiv))

    all_ok = True
    for name, kernel in todo:
        all_ok &= run_version(name, kernel, d_x, d_partial, d_out, x, n, block, grid, atomic_stage2, ref)
    sys.exit(0 if all_ok else 2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile histogram.cu
/*
 * histogram.cu — Histograma de n valores en [0, BINS) en la GPU (Unidad V, S03; conecta con la Práctica 4).
 *
 *   global   cada hilo hace atomicAdd(&hist[v[i]], 1) sobre el histograma en memoria GLOBAL:
 *            n atómicas que compiten por las mismas BINS direcciones (contención alta si los datos
 *            están sesgados: aquí ≈ 30 % cae en el bin 0: 25 % forzado por i % 4 == 0 más lo que
 *            el hash reparte en ese bin).
 *   shared   TODO(1) (fase Crear): histograma PRIVADO por bloque en memoria compartida con atómicas
 *            baratas, y al final un volcado de BINS atómicas por bloque al histograma global.
 *            Es la misma idea que los histogramas privados por hilo + reducción de la Práctica 4.
 *            Mientras no esté completo, el kernel no escribe nada y el programa imprime "TODO pendiente".
 *
 * Uso:  ./histogram [version: global|shared|all] [n] [block] [archivo.csv]     (por defecto all 1048576 256)
 * Datos (iguales en histogram_cpu.c y histogram_numba.py):
 *   v[i] = 0                                        si i % 4 == 0        (sesgo: contención en el bin 0)
 *   v[i] = (((uint32_t)i * 2654435761u) >> 24) % BINS  en otro caso     (hash multiplicativo, pseudoaleatorio)
 * Verificación: los conteos son enteros y deben coincidir EXACTAMENTE con la CPU (aquí no hay redondeo).
 *
 * Compilar y correr (GPU): nvcc -O2 -arch=sm_75 histogram.cu -o histogram && ./histogram
 * Sin GPU: nvcc -c -arch=sm_75 histogram.cu -o histogram.o        (solo comprueba que compila)
 * Otro número de bins (p. ej. bytes): nvcc -O2 -arch=sm_75 -DBINS=256 histogram.cu -o histogram256
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <cuda_runtime.h>

#ifndef BINS
#define BINS 16
#endif
#define REPS 5

#define CUDA_CHECK(call)                                                           \
    do {                                                                           \
        cudaError_t err__ = (call);                                                \
        if (err__ != cudaSuccess) {                                                \
            fprintf(stderr, "Error de CUDA en %s:%d: %s\n", __FILE__, __LINE__,   \
                    cudaGetErrorString(err__));                                    \
            exit(EXIT_FAILURE);                                                    \
        }                                                                          \
    } while (0)

/* ---------- Código de dispositivo ---------- */

/* Versión global: correcta; todos los hilos de todos los bloques golpean las mismas BINS direcciones. */
__global__ void hist_global_atomic(const unsigned char *v, unsigned int *hist, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        atomicAdd(&hist[v[i]], 1u);
}

/* TODO(1): histograma privado por bloque en memoria compartida.
 * Objetivo: mismo resultado que hist_global_atomic con BINS * gridDim.x atómicas globales en lugar de n.
 * Pistas en practica-guiada/README.md, sección "Crear". Supón BINS <= blockDim.x.
 * Mientras el cuerpo esté vacío, el anfitrión detecta el histograma en ceros e imprime "TODO pendiente". */
__global__ void hist_shared_atomic(const unsigned char *v, unsigned int *hist, int n)
{
    (void)v;
    (void)hist;
    (void)n;
}

/* ---------- Código de anfitrión ---------- */

static void init_data(unsigned char *v, int n)
{
    for (int i = 0; i < n; i++) {
        uint32_t h = ((uint32_t)i * 2654435761u) >> 24;   /* hash multiplicativo de Knuth; 0..255 */
        v[i] = (i % 4 == 0) ? 0 : (unsigned char)(h % BINS);
    }
}

static void histogram_cpu(const unsigned char *v, unsigned int *hist, int n)
{
    memset(hist, 0, BINS * sizeof(unsigned int));
    for (int i = 0; i < n; i++)
        hist[v[i]]++;
}

static int cmp_float(const void *a, const void *b)
{
    float fa = *(const float *)a, fb = *(const float *)b;
    return (fa > fb) - (fa < fb);
}

static float median_ms(float *t, int k)
{
    qsort(t, (size_t)k, sizeof(float), cmp_float);
    return t[k / 2];
}

static void print_hist(const char *label, const unsigned int *hist)
{
    printf("%s=[", label);
    for (int b = 0; b < BINS; b++)
        printf("%u%s", hist[b], (b + 1 < BINS) ? "," : "");
    printf("]\n");
}

/* Regresa 1 si la versión pasa (o si está pendiente), 0 si está mal. */
static int run_version(const char *version, const unsigned char *d_v, unsigned int *d_hist, int n, int block,
                       int grid, const unsigned int *h_ref, const char *gpu_name, const char *csv_path)
{
    int is_global = strcmp(version, "global") == 0;
    unsigned int h_hist[BINS];
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float t_kernel[REPS];
    for (int r = -1; r < REPS; r++) {           /* r = -1: calentamiento */
        CUDA_CHECK(cudaMemset(d_hist, 0, BINS * sizeof(unsigned int)));
        CUDA_CHECK(cudaEventRecord(start));
        if (is_global)
            hist_global_atomic<<<grid, block>>>(d_v, d_hist, n);
        else
            hist_shared_atomic<<<grid, block>>>(d_v, d_hist, n);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventSynchronize(stop));
        if (r >= 0)
            CUDA_CHECK(cudaEventElapsedTime(&t_kernel[r], start, stop));
    }
    CUDA_CHECK(cudaMemcpy(h_hist, d_hist, BINS * sizeof(unsigned int), cudaMemcpyDeviceToHost));

    unsigned long long total = 0;
    for (int b = 0; b < BINS; b++)
        total += h_hist[b];
    int exact = memcmp(h_hist, h_ref, BINS * sizeof(unsigned int)) == 0;
    int pending = (total == 0);
    long long atomics_global = is_global ? n : (long long)BINS * grid;
    long long atomics_shared = is_global ? 0 : n;

    float k_med = median_ms(t_kernel, REPS);
    printf("version=%-7s atomicas_globales=%-8lld atomicas_compartidas=%-8lld kernel_mediana_ms=%.4f (min %.4f, max %.4f)\n",
           version, atomics_global, atomics_shared, k_med, t_kernel[0], t_kernel[REPS - 1]);
    print_hist("   hist_gpu", h_hist);
    if (pending)
        printf("   total=0 -> TODO pendiente: hist_shared_atomic no escribe nada todavía (TODO(1))\n");
    else
        printf("   total=%llu verificacion=%s\n", total, exact ? "OK" : "FALLA");

    if (csv_path && !pending) {
        FILE *f = fopen(csv_path, "a");
        if (f) {
            fseek(f, 0, SEEK_END);
            if (ftell(f) == 0)
                fprintf(f, "gpu,version,bins,n,block,grid,atomicas_globales,atomicas_compartidas,"
                           "kernel_mediana_ms,kernel_min_ms,kernel_max_ms,verificacion\n");
            fprintf(f, "\"%s\",%s,%d,%d,%d,%d,%lld,%lld,%.4f,%.4f,%.4f,%s\n", gpu_name, version, BINS, n, block, grid,
                    atomics_global, atomics_shared, k_med, t_kernel[0], t_kernel[REPS - 1], exact ? "OK" : "FALLA");
            fclose(f);
        }
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return pending || exact;
}

int main(int argc, char **argv)
{
    const char *version = (argc > 1) ? argv[1] : "all";
    int n = (argc > 2) ? atoi(argv[2]) : (1 << 20);
    int block = (argc > 3) ? atoi(argv[3]) : 256;
    const char *csv_path = (argc > 4) ? argv[4] : NULL;
    int run_global = strcmp(version, "all") == 0 || strcmp(version, "global") == 0;
    int run_shared = strcmp(version, "all") == 0 || strcmp(version, "shared") == 0;

    if ((!run_global && !run_shared) || n <= 0 || block < BINS || block > 1024) {
        fprintf(stderr, "uso: %s [global|shared|all] [n>0] [block %d..1024] [archivo.csv]\n", argv[0], BINS);
        return 1;
    }
    int grid = (n + block - 1) / block;

    unsigned char *h_v = (unsigned char *)malloc((size_t)n);
    if (!h_v) {
        fprintf(stderr, "Sin memoria en el anfitrión para n=%d\n", n);
        return 1;
    }
    init_data(h_v, n);
    unsigned int h_ref[BINS];
    histogram_cpu(h_v, h_ref, n);

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("histogram (CUDA C): n=%d bins=%d block=%d grid=%d gpu=\"%s\" (SMs=%d, cc %d.%d)\n", n, BINS, block, grid,
           prop.name, prop.multiProcessorCount, prop.major, prop.minor);
    print_hist("hist_cpu", h_ref);

    unsigned char *d_v = NULL;
    unsigned int *d_hist = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_v, (size_t)n));
    CUDA_CHECK(cudaMalloc((void **)&d_hist, BINS * sizeof(unsigned int)));
    CUDA_CHECK(cudaMemcpy(d_v, h_v, (size_t)n, cudaMemcpyHostToDevice));

    int all_ok = 1;
    if (run_global)
        all_ok &= run_version("global", d_v, d_hist, n, block, grid, h_ref, prop.name, csv_path);
    if (run_shared)
        all_ok &= run_version("shared", d_v, d_hist, n, block, grid, h_ref, prop.name, csv_path);

    CUDA_CHECK(cudaFree(d_v));
    CUDA_CHECK(cudaFree(d_hist));
    free(h_v);
    return all_ok ? 0 : 2;
}

In [ ]:
%%writefile histogram_cpu.c
/*
 * histogram_cpu.c — Referencia secuencial de histogram.cu (Unidad V, S03).
 *
 * Imprime el histograma exacto (enteros: debe coincidir bit a bit con la GPU), el sesgo de los
 * datos (fracción del bin más poblado: es la contención que sufre la versión con atómicas globales)
 * y cuántas atómicas globales haría cada versión de GPU con ese n y ese block (fase Predecir).
 *
 * Uso:   ./histogram_cpu [n] [block]        (por defecto 1048576 256)
 * Compilar: gcc -std=c11 -O2 -Wall -Wextra histogram_cpu.c -o histogram_cpu
 * Otro número de bins: gcc ... -DBINS=256 histogram_cpu.c -o histogram_cpu256
 */
#define _POSIX_C_SOURCE 199309L   /* clock_gettime con -std=c11 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <time.h>

#ifndef BINS
#define BINS 16
#endif
#define REPS 5

/* Misma fórmula que histogram.cu y histogram_numba.py */
static void init_data(unsigned char *v, int n)
{
    for (int i = 0; i < n; i++) {
        uint32_t h = ((uint32_t)i * 2654435761u) >> 24;
        v[i] = (i % 4 == 0) ? 0 : (unsigned char)(h % BINS);
    }
}

static void histogram(const unsigned char *v, unsigned int *hist, int n)
{
    memset(hist, 0, BINS * sizeof(unsigned int));
    for (int i = 0; i < n; i++)
        hist[v[i]]++;
}

static double now_ms(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec * 1e3 + (double)ts.tv_nsec / 1e6;
}

static int cmp_double(const void *a, const void *b)
{
    double da = *(const double *)a, db = *(const double *)b;
    return (da > db) - (da < db);
}

int main(int argc, char **argv)
{
    int n = (argc > 1) ? atoi(argv[1]) : (1 << 20);
    int block = (argc > 2) ? atoi(argv[2]) : 256;
    if (n <= 0 || block <= 0) {
        fprintf(stderr, "uso: %s [n>0] [block>0]\n", argv[0]);
        return 1;
    }
    int grid = (n + block - 1) / block;

    unsigned char *v = (unsigned char *)malloc((size_t)n);
    if (!v) {
        fprintf(stderr, "Sin memoria para n=%d\n", n);
        return 1;
    }
    init_data(v, n);

    unsigned int hist[BINS];
    double t[REPS];
    histogram(v, hist, n);                      /* calentamiento */
    for (int r = 0; r < REPS; r++) {
        double t0 = now_ms();
        histogram(v, hist, n);
        t[r] = now_ms() - t0;
    }
    qsort(t, REPS, sizeof(double), cmp_double);

    unsigned long long total = 0;
    unsigned int max_count = 0;
    int max_bin = 0;
    for (int b = 0; b < BINS; b++) {
        total += hist[b];
        if (hist[b] > max_count) {
            max_count = hist[b];
            max_bin = b;
        }
    }

    printf("histogram_cpu: n=%d bins=%d block=%d grid=%d  mediana_ms=%.3f de %d (min %.3f, max %.3f)\n", n, BINS,
           block, grid, t[REPS / 2], REPS, t[0], t[REPS - 1]);
    printf("hist_cpu=[");
    for (int b = 0; b < BINS; b++)
        printf("%u%s", hist[b], (b + 1 < BINS) ? "," : "");
    printf("]\n");
    printf("total=%llu  bin_mas_poblado=%d (%.1f %% de los datos: esa es la contención de la versión global)\n",
           total, max_bin, 100.0 * (double)max_count / (double)n);
    printf("atomicas_globales: global=%d  shared=%d (= %d bins x %d bloques; además %d atómicas en compartida)\n", n,
           BINS * grid, BINS, grid, n);
    printf("verificacion=%s\n", (total == (unsigned long long)n) ? "OK" : "FALLA");

    free(v);
    return (total == (unsigned long long)n) ? 0 : 2;
}

In [ ]:
%%writefile histogram_numba.py
#!/usr/bin/env python3
"""histogram_numba.py — Espejo en Numba CUDA de histogram.cu.

Uso:
    python3 histogram_numba.py [version: global|shared|all] [n] [block] [--bins B]
Sin GPU (simulador, solo correctitud; n pequeño):
    NUMBA_ENABLE_CUDASIM=1 python3 histogram_numba.py global 4096 256

Mismos datos que histogram.cu / histogram_cpu.c (hash multiplicativo; ≈ 30 % de los datos en el bin 0:
25 % forzado por i % 4 == 0 más lo que el hash reparte ahí) y
verificación EXACTA contra numpy.bincount (conteos enteros: no hay redondeo).
La versión `shared` es el TODO(1) de la fase Crear: mientras el kernel no escriba nada, se imprime
"TODO pendiente".
"""
import argparse
import os
import sys
import time

import numpy as np
from numba import cuda, uint32

CUDASIM = os.environ.get("NUMBA_ENABLE_CUDASIM", "0") == "1"
REPS = 5


@cuda.jit
def hist_global_atomic(v, hist, n):
    # Todos los hilos de todos los bloques compiten por las mismas `bins` direcciones globales.
    i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if i < n:
        cuda.atomic.add(hist, v[i], 1)


def make_shared_kernel(bins):
    """`bins` dimensiona el arreglo en memoria compartida; por eso se captura en la clausura."""

    @cuda.jit
    def hist_shared_atomic(v, hist, n):
        # TODO(1): histograma privado por bloque en cuda.shared.array(bins, uint32):
        # poner en cero, syncthreads, atómicas en compartida, syncthreads, volcado con `bins` atómicas
        # globales por bloque. Pistas en README.md, sección "Crear". Mientras tanto no escribe nada.
        i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
        if i < 0:                      # nunca cierto: el kernel queda vacío a propósito
            hist[0] = uint32(0)

    return hist_shared_atomic


def init_data(n, bins):
    i = np.arange(n, dtype=np.uint64)
    h = ((i * np.uint64(2654435761)) & np.uint64(0xFFFFFFFF)) >> np.uint64(24)
    v = (h % np.uint64(bins)).astype(np.uint8)
    v[::4] = 0                                   # sesgo: i % 4 == 0 -> bin 0
    return v


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("version", nargs="?", default="all", choices=["global", "shared", "all"])
    parser.add_argument("n", nargs="?", type=int, default=(4096 if CUDASIM else 1 << 20))
    parser.add_argument("block", nargs="?", type=int, default=256)
    parser.add_argument("--bins", type=int, default=16)
    args = parser.parse_args()
    n, block, bins = args.n, args.block, args.bins
    if n <= 0 or block < bins or block > 1024:
        sys.exit("uso: histogram_numba.py [version] [n>0] [block bins..1024] [--bins B]")
    grid = (n + block - 1) // block

    v = init_data(n, bins)
    ref = np.bincount(v, minlength=bins).astype(np.uint32)
    d_v = cuda.to_device(v)
    d_hist = cuda.to_device(np.zeros(bins, dtype=np.uint32))

    print(f"histogram_numba: n={n} bins={bins} block={block} grid={grid} modo={'CUDASIM' if CUDASIM else 'GPU'}")
    print("hist_cpu=[" + ",".join(str(c) for c in ref) + "]")

    versions = []
    if args.version in ("all", "global"):
        versions.append(("global", hist_global_atomic, n, 0))
    if args.version in ("all", "shared"):
        versions.append(("shared", make_shared_kernel(bins), bins * grid, n))

    all_ok = True
    for name, kernel, atomics_global, atomics_shared in versions:
        times_ms = []
        reps = range(0, 1) if CUDASIM else range(-1, REPS)   # CUDASIM: un solo lanzamiento
        for r in reps:
            d_hist.copy_to_device(np.zeros(bins, dtype=np.uint32))
            t0 = time.perf_counter()
            kernel[grid, block](d_v, d_hist, n)
            cuda.synchronize()
            if r >= 0:
                times_ms.append((time.perf_counter() - t0) * 1e3)
        hist = d_hist.copy_to_host()
        total = int(hist.sum())
        exact = np.array_equal(hist, ref)
        timing = (f"tiempo_simulador_ms={times_ms[0]:.1f} (NO es una medición)" if CUDASIM else
                  f"kernel_mediana_ms={sorted(times_ms)[len(times_ms) // 2]:.4f} "
                  f"(min {min(times_ms):.4f}, max {max(times_ms):.4f})")
        print(f"version={name:<7} atomicas_globales={atomics_global:<8} atomicas_compartidas={atomics_shared:<8} {timing}")
        print("   hist_gpu=[" + ",".join(str(c) for c in hist) + "]")
        if total == 0:
            print("   total=0 -> TODO pendiente: hist_shared_atomic no escribe nada todavía (TODO(1))")
        else:
            print(f"   total={total} verificacion={'OK' if exact else 'FALLA'}")
            all_ok &= exact
    sys.exit(0 if all_ok else 2)


if __name__ == "__main__":
    main()

## 2. Compilar

En una T4 la línea equivale a `nvcc -O2 -arch=sm_75 reduce_sum.cu -o reduce_sum`. Las referencias CPU se compilan con `gcc -Wall -Wextra` (sin *warnings*).

In [ ]:
# En una T4 (Colab) esto equivale a:  !nvcc -O2 -arch=sm_75 reduce_sum.cu -o reduce_sum
!nvcc -O2 -arch={ARCH} reduce_sum.cu -o reduce_sum
!nvcc -O2 -arch={ARCH} histogram.cu -o histogram
!gcc -std=c11 -O2 -Wall -Wextra reduce_sum_cpu.c -o reduce_sum_cpu -lm
!gcc -std=c11 -O2 -Wall -Wextra histogram_cpu.c -o histogram_cpu
!ls -l reduce_sum histogram reduce_sum_cpu histogram_cpu

## 3. Ejecutar (fase *Run* de PRIMM)

Primero la referencia CPU (cuatro órdenes de suma distintos, mismo dato); después las tres versiones en GPU con `n = 2^20` y bloques de 256. Compara `suma`, `err_rel`, `atomicas_globales` y `kernel_mediana_ms` con lo que predijiste.

In [ ]:
!./reduce_sum_cpu 1048576 256

In [ ]:
!./reduce_sum all 1048576 256 cpu

In [ ]:
# Segunda etapa en la GPU: una atómica por bloque en lugar de copiar los parciales al anfitrión
!./reduce_sum tree_nodiv 1048576 256 atomic

El espejo en Numba hace lo mismo desde Python (en GPU real la primera llamada compila el kernel; por eso calienta). Sin GPU, en tu laptop: `NUMBA_ENABLE_CUDASIM=1 python3 reduce_sum_numba.py all 1000 256`.

In [ ]:
!python3 reduce_sum_numba.py all 1048576 256

## 4. Investigar

**a) Sin `__syncthreads()` entre niveles** (`-DSIN_SYNC`). Predice: ¿sale mal, o sale bien "por suerte"? Corre cinco veces con `block = 256` y cinco con `block = 32` (un solo *warp*). Anota cuántas veces dio `OK` cada una.

**b) `n` no potencia de 2 ni múltiplo del bloque** (`n = 1 000 003`): el último bloque carga ceros donde `i ≥ n`.

**c) `block = 96` (no potencia de 2)**: no hay error de CUDA. ¿Qué dice la verificación? ¿Qué fracción de los datos se perdió?

**d) `n = 2^24` con la versión atómica**: el acumulador `float` llega a 16 777 216, donde sumar 1.0 ya no cambia el valor. ¿Qué versión falla y cuál no?

In [ ]:
!nvcc -O2 -arch={ARCH} -DSIN_SYNC reduce_sum.cu -o reduce_sum_sin_sync
for block in (256, 32):
    print(f"=== -DSIN_SYNC, block={block}: 5 corridas ===")
    for k in range(5):
        out = subprocess.run(["./reduce_sum_sin_sync", "tree_nodiv", "1048576", str(block)],
                             capture_output=True, text=True).stdout
        print([l.strip() for l in out.splitlines() if "verificacion=" in l][0])

In [ ]:
!./reduce_sum all 1000003 256; echo "código de salida: $?"

In [ ]:
!./reduce_sum tree_nodiv 1048576 96; echo "código de salida: $?"

In [ ]:
!./reduce_sum_cpu 16777216 256
!./reduce_sum all 16777216 256; echo "código de salida: $?"

## 5. Medir (protocolo del curso)

`reduce_sum` ya hace calentamiento + 5 repeticiones y reporta la mediana. Aquí barremos las tres versiones para varios `n` y las dos formas de la segunda etapa, y guardamos cada fila en `resultados.csv`. Anota el modelo de GPU y la fecha: el hardware que asigna Colab cambia. Los tiempos de la T4 son órdenes de magnitud; lo que debe coincidir es `suma`, `err_rel` y `atomicas_globales`.

In [ ]:
import os, subprocess
if os.path.exists("resultados.csv"):
    os.remove("resultados.csv")
for n in [1 << 16, 1 << 20, 1 << 22]:
    for stage2 in ["cpu", "atomic"]:
        out = subprocess.run(["./reduce_sum", "all", str(n), "256", stage2, "resultados.csv"],
                             capture_output=True, text=True)
        for line in out.stdout.splitlines():
            if line.startswith("version="):
                print(f"n={n:>8} " + line)
        if out.returncode != 0:
            print("código de salida:", out.returncode, out.stderr)

## 6. Histograma con atómicas (fase Crear)

`histogram global` hace `atomicAdd` sobre 16 contadores en memoria global desde todos los hilos; el 30 % de los datos cae en el bin 0 (contención). `histogram shared` es el `TODO(1)`: histograma privado por bloque en memoria compartida y volcado de 16 atómicas por bloque. Mientras no lo completes, imprime `TODO pendiente`. Cuando lo completes: edita la celda `%%writefile histogram.cu`, vuelve a ejecutarla, recompila y corre esta celda; el criterio de "listo" es `verificacion=OK` con el mismo histograma que la CPU y `atomicas_globales=65536` para `n = 2^20`.

In [ ]:
!./histogram_cpu 1048576 256
!./histogram all 1048576 256 resultados_histograma.csv

In [ ]:
# Con 256 bins (bytes): misma idea, 16x más atómicas por bloque en el volcado
!nvcc -O2 -arch={ARCH} -DBINS=256 histogram.cu -o histogram256
!./histogram256 all 1048576 256

## 7. Guardar resultados

La celda final muestra la tabla, agrega la fecha y descarga el CSV (en Kaggle queda en `/kaggle/working/`). Súbelo a tu repositorio en `resultados/`. Para tu bitácora: ¿cuántas veces más tarda la versión atómica que el árbol? ¿Cambió `suma` entre corridas de la versión atómica? ¿Y entre las dos del árbol?

In [ ]:
import datetime
import pandas as pd
df = pd.read_csv("resultados.csv")
df["fecha"] = datetime.date.today().isoformat()
print(df.to_string(index=False))
nombre = f"resultados_S03_reduccion_{datetime.date.today().isoformat()}.csv"
df.to_csv(nombre, index=False)
print("guardado:", nombre)
if os.path.exists("resultados_histograma.csv"):
    dh = pd.read_csv("resultados_histograma.csv")
    dh["fecha"] = datetime.date.today().isoformat()
    print(dh.to_string(index=False))
    nombre_h = f"resultados_S03_histograma_{datetime.date.today().isoformat()}.csv"
    dh.to_csv(nombre_h, index=False)
    print("guardado:", nombre_h)
try:
    from google.colab import files  # solo existe en Colab
    files.download(nombre)
except ImportError:
    print("En Kaggle: descarga el archivo desde el panel Output (/kaggle/working/).")